In [ ]:
!pip install enoslib ipywidgets

In [1]:
!ssh rennes.grid5000.fr hostname

frennes


### Building and pushing the docker images to dockerhub

In [2]:
!/home/corentin/fcquic_applications_master_thesis/docker_images/build_images.sh

Building base image
[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.2s (2/3)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 1.75kB                                     0.0s
 => [internal] load metadata for docker.io/library/debian:bookworm-slim    0.2s
 => [auth] library/debian:pull token for registry-1.docker.io              0.0s
[+] Building 0.3s (2/3)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 1.75kB                                     0.0s
 => [internal] load metadata for docker.io/library/debian:bookworm-slim    0.3s
 => [auth] library/debian:pull token for registry-1.docker.io              0.0s
[+] Building 0.5s (2/3)                                          docker:default
 => [internal] load 

In [3]:
!/home/corentin/fcquic_applications_master_thesis/docker_images/upload_images.sh

tagging and pushing base


The push refers to repository [docker.io/corentindetry/base]

b238bc9a: Waiting 
53f2dec3: Waiting 
12bd67ef: Waiting 
ebc9fdcb: Waiting 
1b5e9776: Waiting 
fca4d5c8: Waiting 
17e0a25e: Waiting 
890ea33a: Waiting 
596775ab: Waiting 
1c60fe54: Waiting 
afebaf4d: Waiting 
20a133dd: Waiting 
bc9fdcb: Pushed   3.049MB/3.049MBPushing  1.252kB/1.252kBPushing  3.061MB/3.061MBlatest: digest: sha256:ea29f16328655473ae5c5340525e5823e054e78b133fc39c7ea3f30f014d01d6 size: 856
tagging and pushing fcquic_client
The push refers to repository [docker.io/corentindetry/fcquic_client]

17e0a25e: Waiting 
596775ab: Waiting 
afebaf4d: Waiting 
fca4d5c8: Waiting 
1b5e9776: Waiting 
b238bc9a: Waiting 
53f2dec3: Waiting 
ebc9fdcb: Waiting 
be19fd1f: Waiting 
12bd67ef: Waiting 
1c60fe54: Waiting 
819edcf2: Waiting 
fe8e9fb7: Waiting 
bd7beca0: Waiting 
bd7beca0: Pushed   1.287kB/1.287kBry/base Klatest: digest: sha256:3bfd03e12a91f74428cc84d1382e7c2ab053b1436351081e3914ce4b4b80d590 size: 856
tagging and pushing

### G5K connection

Setup the `.python-grid5000.yaml` file with the username and password used to login to grid5000.

The file should look like this:
```yaml
username: G5K_LOGIN
password: G5K_password
```

-> required in order for the enoslib calls to g5k to work.

In addition to this, make sure to add these lines to your ssh configuration:

```text
Host g5k
    User G5K_LOGIN
    HostName access.grid5000.fr
    ForwardAgent no

Host !access.grid5000.fr *.grid5000.fr
    User G5K_LOGIN
    ProxyJump G5K_LOGIN@access.grid5000.fr
    StrictHostKeyChecking no
    UserKnownHostsFile /dev/null
    ForwardAgent yes

Host access.grid5000.fr
    User G5K_LOGIN
    StrictHostKeyChecking no
    UserKnownHostsFile /dev/null
    ForwardAgent yes

Host *.g5k
    User G5K_LOGIN
    ProxyCommand ssh g5k -W "$(basename %h .g5k):%p"
    ForwardAgent no
```

Make sure to replace `G5K_LOGIN` with your g5k username (the one from the site)

In [4]:
import os
from grid5000 import Grid5000

conf_file = os.path.join(os.environ.get("HOME"), ".python-grid5000.yaml")
gk = Grid5000.from_yaml(conf_file)

print("Sites: ")
display(gk.sites.list())

Sites: 


[<Site uid:bordeaux>,
 <Site uid:grenoble>,
 <Site uid:lille>,
 <Site uid:louvain>,
 <Site uid:luxembourg>,
 <Site uid:lyon>,
 <Site uid:nancy>,
 <Site uid:nantes>,
 <Site uid:rennes>,
 <Site uid:sophia>,
 <Site uid:strasbourg>,
 <Site uid:toulouse>]

### Job configuration

Setup the various parameters for the job:
- The job name will identify the current booking, if the notebook kernel dies, re running the same reservation code with the same name will reload the existing job instead of booking a new one
- The walltime is the time that the booking will last, you can always stop your reservation earlier than the booking's end time

In [5]:
JOB_NAME="fcquic_multisite_test"
JOB_WALLTIME="0:10:00"

### Create the booking

This first case will only create the booking object locally, but not yet place the booking.

You can adjust the different roles of the machines to match the roles defined in the NPF script.

For example, here I have 2 roles: server and client. The server node will run the server section of the NPF script,...

In [6]:
import enoslib as en
from npf import enoslib as npf
from npf.output.transform.pandas import to_pandas

from importlib import reload
reload(npf)

conf = (
    en.G5kConf.from_settings(job_name=JOB_NAME, walltime=JOB_WALLTIME)
    # For convenience, we use the site name as role
    .add_machine(roles=["louvain"], cluster="spirou", nodes=1)
    .add_machine(roles=["nancy"], cluster="gros", nodes=1)
)

provider = en.G5k(conf)

[WARNING]: failed to patch stdout/stderr for fork-safety: 'OutStream' object
has no attribute 'buffer'
[WARNING]: failed to reconfigure stdout/stderr with custom encoding error
handler: 'OutStream' object has no attribute 'reconfigure'


In [7]:
print("Reserving resources...")

# Get actual resources
roles, networks = provider.init()
display(roles)
display(networks)

Reserving resources...


Output()

Finished 1 tasks (Granting root access on the nodes (sudo-g5k)) on 
{'spirou-5.louvain.grid5000.fr', 'gros-9.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

{'louvain': {Host(address='spirou-5.louvain.grid5000.fr', alias='spirou-5.louvain.grid5000.fr', user='root', keyfile=None, port=None, extra={'gateway': 'access.grid5000.fr', 'gateway_user': 'cdetry'}, net_devices=set(), _Host__original_extra={'gateway': 'access.grid5000.fr', 'gateway_user': 'cdetry'})}, 'nancy': {Host(address='gros-9.nancy.grid5000.fr', alias='gros-9.nancy.grid5000.fr', user='root', keyfile=None, port=None, extra={'gateway': 'access.grid5000.fr', 'gateway_user': 'cdetry'}, net_devices=set(), _Host__original_extra={'gateway': 'access.grid5000.fr', 'gateway_user': 'cdetry'})}}

[G5k] gateway is not yet implemented for <class 'enoslib.infra.enos_g5k.objects.G5kEnosProd6Network'> on the G5k side
[G5k] gateway is not yet implemented for <class 'enoslib.infra.enos_g5k.objects.G5kEnosProd6Network'> on the G5k side


{'prod': {<enoslib.infra.enos_g5k.objects.G5kEnosProd4Network object at 0x7205f8971e20>, <enoslib.infra.enos_g5k.objects.G5kEnosProd4Network object at 0x7205f8a02420>, <enoslib.infra.enos_g5k.objects.G5kEnosProd6Network object at 0x7205f8a02c60>, <enoslib.infra.enos_g5k.objects.G5kEnosProd6Network object at 0x7205f8a6d940>}}

In [8]:
# From: https://discovery.gitlabpages.inria.fr/enoslib/tutorials/grid5000.html#kavlan-on-secondary-interfaces
# Fill in network information from nodes
roles = en.sync_info(roles, networks)

Output()

Finished 1 tasks (Waiting for connection) on {'spirou-5.louvain.grid5000.fr', 
'gros-9.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

Finished 7 tasks (Gathering Facts,setup,utils : include_tasks,utils : Dump network 
information in a file,utils : Create the fake interfaces) on {'spirou-5.louvain.grid5000.fr',
'gros-9.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

In [11]:
data = {}
for node_role, hosts_data in roles.items():
    display(host_data)
    for host_data in hosts_data:

        # Get each node's IP address on the private network
        ip_address_obj = host_data.filter_addresses(networks=networks["private"])[0]
        # This may seem weird: ip_address_obj.ip is a `netaddr.IPv4Interface`
        # which itself has an `ip` attribute.
        node_ip = ip_address_obj.ip.ip
        if data.get(node_role) is None:
           data[node_role] = []
        data[node_role].append(node_ip.exploded)


display(data)
# print(f"Server IP: {data["server"]}")

ip
127.0.0.1/8 # noqa
::1/128 # noqa
ip
fe80::eaeb:d3ff:fefd:cf84/64 # noqa
172.16.208.5/20 # noqa


IndexError: list index out of range

In [13]:

# Check connectivity from Rennes to Lille
target = roles["louvain"][0]
results = en.run_command(f"ping -c10 {target.address}", roles=roles["nancy"])
for result in results:
    print(f"Ping from {result.host} to {target.address}:")
    print(f"{result.stdout}")


Output()

Finished 1 tasks (ping -c10 spirou-5.louvain.grid5000.fr) on {'gros-9.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Ping from gros-9.nancy.grid5000.fr to spirou-5.louvain.grid5000.fr:
PING spirou-5.louvain.grid5000.fr (172.16.208.5) 56(84) bytes of data.
64 bytes from spirou-5.louvain.grid5000.fr (172.16.208.5): icmp_seq=1 ttl=62 time=12.0 ms
64 bytes from spirou-5.louvain.grid5000.fr (172.16.208.5): icmp_seq=2 ttl=62 time=12.0 ms
64 bytes from spirou-5.louvain.grid5000.fr (172.16.208.5): icmp_seq=3 ttl=62 time=12.0 ms
64 bytes from spirou-5.louvain.grid5000.fr (172.16.208.5): icmp_seq=4 ttl=62 time=12.0 ms
64 bytes from spirou-5.louvain.grid5000.fr (172.16.208.5): icmp_seq=5 ttl=62 time=12.0 ms
64 bytes from spirou-5.louvain.grid5000.fr (172.16.208.5): icmp_seq=6 ttl=62 time=12.0 ms
64 bytes from spirou-5.louvain.grid5000.fr (172.16.208.5): icmp_seq=7 ttl=62 time=12.0 ms
64 bytes from spirou-5.louvain.grid5000.fr (172.16.208.5): icmp_seq=8 ttl=62 time=12.0 ms
64 bytes from spirou-5.louvain.grid5000.fr (172.16.208.5): icmp_seq=9 ttl=62 time=12.0 ms
64 bytes from spirou-5.louvain.grid5000.fr (172.16.

In [14]:
# Release all Grid'5000 resources
provider.destroy()